In [1]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
import io

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}

In [5]:
def get_html(url):
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    return resp.text

In [11]:
def get_mvp_table(year):

    url = f"https://www.basketball-reference.com/awards/awards_{year}.html"
    html = get_html(url)

    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", {"id": "mvp"})

    df = pd.read_html(io.StringIO(str(table)))[0]

    # the header on basketball-reference has two rows, remove first one :)
    df.columns = df.columns.droplevel(0)

    # get player_id
    player_id_list = []
    rows = table.find("tbody").find_all("tr")

    for row in rows:
        player_cell = row.find("td", {"data-stat": "player"})
        if player_cell is not None:
            player_id_list.append(player_cell.get("data-append-csv"))
        else:
            player_id_list.append(None)

    if len(player_id_list) == len(df):
        df["player_id"] = player_id_list
    else:
        print("Warning: length mismatch for year", year)

    # rename Tm to Team to be clearer
    df = df.rename(columns={"Tm": "Team"})

    # season like "2019-20"
    df["season"] = str(year - 1) + "-" + str(year)[2:]

    return df

In [13]:
all_tables = []

for year in range(2020, 2025):
    print("Year:", year)
    df = get_mvp_table(year)
    print(len(df), "rows")
    all_tables.append(df)
    time.sleep(4)

final_df = pd.concat(all_tables)
final_df.to_csv("mvp_voting.csv", index=False)

Year: 2020
12 rows
Year: 2021
15 rows
Year: 2022
12 rows
Year: 2023
13 rows
Year: 2024
9 rows
